# Epidemiological models

This notebook is the summer4 equivalent of summer2's infection and mixing
pages (`examples/04-flow-types`, `examples/09-mixing-matrices`). It is written
for epidemiologists who already think in terms of susceptibles, infectious
people, contact rates, and who-mixes-with-whom.

We will:

1. Build a plain SIR epidemic and plot it.
2. Contrast **frequency-dependent** and **density-dependent** transmission.
3. Stratify by age and attach a **mixing matrix**.
4. Show why homogeneous mixing makes age-specific infectiousness
   unidentifiable — and how assortative mixing fixes that.
5. Use the short `EpiModel` frontend (summer2 muscle memory).

Every plot is accompanied by an assertion so the page stays a runnable test.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    GroupedOutput,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import EpiModel, ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"


def plot_compartments(res, title):
    """Plot S, I, R totals over time (summer2-style outputs plot)."""
    frame = res["comp"].to_pandas()
    # Columns are compartment labels; sum age strata when present.
    totals = {}
    for col in frame.columns:
        state_name = col.split("_")[0] if "_" in col else col
        totals.setdefault(state_name, 0.0)
        totals[state_name] = totals[state_name] + frame[col]
    return pd.DataFrame(totals, index=frame.index).plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


def plot_by_age(res, state_name, title):
    """Plot one disease state, one line per age band."""
    frame = res["comp"].select(state[state_name]).to_pandas()
    frame.columns = [c.split("_")[-1] for c in frame.columns]
    return frame.plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


## A plain SIR epidemic

Start with no age structure. People move $S \rightarrow I$ when they are
infected, and $I \rightarrow R$ when they recover. The infection rate depends
on how many people are currently infectious — that dependence is the
**force of infection** $\lambda$.

Here we use the summer2-shaped frontend: name the infectious compartments once,
then add an infection flow and a recovery flow.


In [ ]:
state = Property("state", ("S", "I", "R"))
# ForceOfInfection always groups by a property; a single-trait "pop" stands in
# for the unstratified whole-population case.
pop = Property("pop", ("all",))
pmap1 = PropertyMap.from_property(state).stratify(pop)

m = EpiModel(pmap1, infectious=state["I"])
m.set_mixing_matrix(pop, np.array([[1.0]]), check_reciprocal=False)
m.add_infection_frequency_flow("infection", state["S"], state["I"], contact_rate=1.0)
m.add_transition_flow("recovery", state["I"], state["R"], 1.0 / 3.0)

y0 = np.zeros(pmap1.size)
y0[pmap1.select(state["S"])] = 990.0
y0[pmap1.select(state["I"])] = 10.0
plan = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 20.0, 201),
)
res = m.compile().run({}, y0, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")

i_peak = float(np.max(np.asarray(res["comp"].select(state["I"]).values.data)))
assert i_peak > 10.0, "epidemic should grow from the seed"
plot_compartments(res, "Unstratified SIR (frequency-dependent infection)")


## Frequency vs density transmission

Two common ways to write the force of infection (summer2's
`add_infection_frequency_flow` / `add_infection_density_flow`):

| Kind | Force of infection $\lambda$ | Typical use |
|---|---|---|
| **Frequency** | $\lambda = c \, I / N$ | Close-contact diseases; rate depends on the *proportion* infectious |
| **Density** | $\lambda = c \, I$ | Environmental / density-driven contact; rate grows with absolute numbers |

The number of new infections per day is then $\lambda \times S$ in both cases.

The practical difference shows up when population size changes: under frequency
dependence, doubling everyone leaves $\lambda$ the same; under density
dependence it doubles. Below we run the same seed epidemic both ways and plot
infectious prevalence side by side.


In [ ]:
def run_kind(kind, contact_rate):
    mm = EpiModel(pmap1, infectious=state["I"])
    mm.set_mixing_matrix(pop, np.array([[1.0]]), check_reciprocal=False)
    if kind == "frequency":
        mm.add_infection_frequency_flow(
            "infection", state["S"], state["I"], contact_rate
        )
    else:
        mm.add_infection_density_flow(
            "infection", state["S"], state["I"], contact_rate
        )
    mm.add_transition_flow("recovery", state["I"], state["R"], 1.0 / 3.0)
    return mm.compile().run({}, y0, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")


freq = run_kind("frequency", contact_rate=1.0)
# Density contact rate is per person, so a small number matches a similar peak.
dens = run_kind("density", contact_rate=1e-3)

i_freq = np.asarray(freq["comp"].select(state["I"]).values.data)[:, 0]
i_dens = np.asarray(dens["comp"].select(state["I"]).values.data)[:, 0]
ts = np.asarray(freq["comp"].times.values)
compare = pd.DataFrame({"frequency": i_freq, "density": i_dens}, index=ts)
assert float(np.max(i_freq)) > 10.0 and float(np.max(i_dens)) > 10.0
compare.plot(
    title="Infectious prevalence: frequency vs density",
    labels={"index": "time (days)", "value": "infectious people"},
)


## Age stratification and who mixes with whom

By default, models assume **homogeneous mixing**: every person contacts every
other person at the same rate. That is often wrong. Children may mix mostly
with other children; older adults may mix mostly among themselves
(**assortative** or with-like mixing).

A **mixing matrix** $K$ is an $N \times N$ table for $N$ strata. Following
summer2: **columns are infectors, rows are the infected**. So $K_{ab}$ is the
relative contact that stratum $a$ (row) receives from stratum $b$ (column).

For two age bands:

| | young (infector) | old (infector) |
|---|---|---|
| **young** (infected) | young → young | old → young |
| **old** (infected) | young → old | old → old |

Under frequency dependence the force of infection in band $a$ is

$$\lambda_a = c \sum_b K_{ab}\,\frac{I_b}{N_b}.$$


In [ ]:
age = Property("age", ("young", "old"))
pmap_age = PropertyMap.from_property(state).stratify(age)

# Mildly assortative: more within-age contact than between-age.
K_assort = np.array(
    [
        [0.8, 0.2],  # young infected by young, old
        [0.3, 0.7],  # old infected by young, old
    ]
)

epi = EpiModel(pmap_age, infectious=state["I"])
epi.set_mixing_matrix(age, K_assort, normalize="rows", check_reciprocal=False)
epi.add_infection_frequency_flow("infection", state["S"], state["I"], Param("beta"))
epi.add_transition_flow("recovery", state["I"], state["R"], 1.0 / 3.0)

y0_age = np.zeros(pmap_age.size)
y0_age[pmap_age.select(state["S"] & age["young"])] = 490.0
y0_age[pmap_age.select(state["S"] & age["old"])] = 490.0
# Seed infection in the young only — assortative mixing keeps more of the
# early wave in that band.
y0_age[pmap_age.select(state["I"] & age["young"])] = 20.0
y0_age[pmap_age.select(state["I"] & age["old"])] = 0.0

plan_age = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "foi": SaveRequest(GroupedOutput("infection")),
    },
    ts=np.linspace(0.0, 40.0, 81),
)
res_age = epi.compile().run(
    {"beta": 1.2}, y0_age, t0=0.0, t1=40.0, dt=0.1, save=plan_age, solver="euler"
)

plot_by_age(res_age, "I", "Infectious people by age (assortative mixing)")


### Inspecting the force of infection itself

summer4 saves $\lambda$ as a properly dimensioned trace — one value per age
band per time — so you can plot and query it without re-slicing a broadcast
array. Under assortative mixing and a young-only seed, $\lambda$ should be
higher in the young early on.


In [ ]:
foi_frame = res_age["foi"].to_pandas()
foi_frame.columns = list(age.traits)
assert res_age["foi"].dims == ("time", "age")
# Early in the epidemic, young λ exceeds old λ under assortative mixing.
early = foi_frame.iloc[5]
assert float(early["young"]) > float(early["old"])
foi_frame.plot(
    title="Force of infection λ by age (assortative mixing)",
    labels={"index": "time (days)", "value": "λ (per day)"},
)


## Why mixing structure matters for inference

A common modelling choice is a **homogeneous** mixing matrix — every entry
equal (after row normalisation, $K_{ab} = 1/N$). Then every age band sees
**exactly the same** force of infection at every time:

$$\lambda_a = c \sum_b \tfrac{1}{N}\,\frac{I_b}{N_b} = \lambda$$

for all $a$. Age-specific infectiousness weights cannot be told apart from a
rescaling of the overall contact rate: the data carry no age gradient in
$\lambda$. An **assortative** matrix restores band-specific $\lambda_a$ and
makes that gradient identifiable again.

The plots below make the contrast visible. We assert that the maximum
difference between age bands is exactly zero under homogeneous mixing, and
strictly positive under assortative mixing.


In [ ]:
def run_mixing(K, title_suffix):
    model = FlowModel(pmap_age)
    foi = ForceOfInfection(
        "infection",
        infectious=state["I"],
        group_by=age,
        kind="frequency",
        contact_rate=Param("beta"),
        mixing=MixingMatrix(age, K, normalize="none", check_reciprocal=False),
    )
    model.add_flow(TransitionFlow("infection", state["S"], state["I"], foi))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
    out = model.compile().run(
        {"beta": 0.8},
        y0_age,
        t0=0.0,
        t1=30.0,
        dt=0.1,
        save=plan_age,
        solver="euler",
    )
    vals = np.asarray(out["foi"].values.data)
    frame = out["foi"].to_pandas()
    frame.columns = list(age.traits)
    diff = float(np.max(np.abs(vals[:, 0] - vals[:, 1])))
    fig = frame.plot(
        title=f"λ by age — {title_suffix} (max |Δλ| = {diff:.4g})",
        labels={"index": "time (days)", "value": "λ (per day)"},
    )
    return out, diff, fig


K_hom = np.ones((2, 2)) / 2.0
K_ass = np.array([[0.9, 0.1], [0.1, 0.9]])

_, diff_hom, fig_hom = run_mixing(K_hom, "homogeneous mixing")
fig_hom


In [ ]:
_, diff_ass, fig_ass = run_mixing(K_ass, "assortative mixing")
assert diff_hom == 0.0, "homogeneous mixing must collapse λ across ages"
assert diff_ass > 0.0, "assortative mixing must produce age-specific λ"
print(f"homogeneous max |Δλ| = {diff_hom}")
print(f"assortative max |Δλ| = {diff_ass:.4f}")
fig_ass


## Age-varying infectiousness

Sometimes one age band is more infectious per infectious person (different
viral load, behaviour, or duration). summer2 put those weights on the
stratification; summer4 attaches them to the force of infection — a band does
not know what a pathogen is.

`normalize="population"` rescales the weights so their population-weighted
mean is 1. That breaks the redundancy between "everyone is twice as
infectious" and "double the contact rate", without needing a prior demography
run to supply population shares.


In [ ]:
epi_nu = EpiModel(pmap_age, infectious=state["I"])
epi_nu.set_mixing_matrix(age, K_ass, normalize="none", check_reciprocal=False)
epi_nu.add_infectiousness_adjustments(
    age, {"young": 0.7, "old": 1.4}, normalize="population"
)
epi_nu.add_infection_frequency_flow("infection", state["S"], state["I"], Param("beta"))
epi_nu.add_transition_flow("recovery", state["I"], state["R"], 1.0 / 3.0)

res_nu = epi_nu.compile().run(
    {"beta": 0.8}, y0_age, t0=0.0, t1=40.0, dt=0.1, save=plan_age, solver="euler"
)

# Same weights ×2 — under population normalisation the trajectories match.
epi_nu2 = EpiModel(pmap_age, infectious=state["I"])
epi_nu2.set_mixing_matrix(age, K_ass, normalize="none", check_reciprocal=False)
epi_nu2.add_infectiousness_adjustments(
    age, {"young": 1.4, "old": 2.8}, normalize="population"
)
epi_nu2.add_infection_frequency_flow("infection", state["S"], state["I"], Param("beta"))
epi_nu2.add_transition_flow("recovery", state["I"], state["R"], 1.0 / 3.0)
res_nu2 = epi_nu2.compile().run(
    {"beta": 0.8}, y0_age, t0=0.0, t1=40.0, dt=0.1, save=plan_age, solver="euler"
)
np.testing.assert_allclose(
    np.asarray(res_nu["comp"].values.data),
    np.asarray(res_nu2["comp"].values.data),
    atol=1e-5,
)
print("2× infectiousness weights ≡ same epidemic under normalize='population'")
plot_by_age(res_nu, "I", "Infectious by age (age-varying infectiousness)")


## The `EpiModel` frontend vs building by hand

Everything above can be written with `ForceOfInfection` and `FlowModel`
directly. `EpiModel` is sugar in the shape of summer2's `CompartmentalModel`:

```python
m = EpiModel(pmap, infectious=state["I"])
m.set_mixing_matrix(age, contacts)
m.add_infectiousness_adjustments(age, {"young": 0.7, "old": 1.4})
m.add_infection_frequency_flow("infection", state["S"], state["I"], Param("beta"))
```

Escape hatches are deliberate: `m.flow_model` is the underlying `FlowModel`,
`m.foi(name)` returns the live `ForceOfInfection`, and `m.add_flow(...)`
accepts any declarative flow. A model built through the frontend compiles to
the **same digest** as the hand-built equivalent — no generality is lost.


In [ ]:
frontend = EpiModel(pmap_age, infectious=state["I"])
frontend.set_mixing_matrix(age, K_ass, normalize="none", check_reciprocal=False)
frontend.add_infection_frequency_flow(
    "infection", state["S"], state["I"], Param("beta")
)
frontend.add_transition_flow("recovery", state["I"], state["R"], 1.0 / 3.0)

declarative = FlowModel(pmap_age)
declarative.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=age,
            mixing=MixingMatrix(age, K_ass, normalize="none", check_reciprocal=False),
            contact_rate=Param("beta"),
        ),
    )
)
declarative.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))

assert frontend.compile() == declarative.compile()
assert frontend.foi("infection").kind == "frequency"
print("EpiModel digest equals declarative ForceOfInfection build")


## Summary

| Idea | summer2 | summer4 |
|---|---|---|
| Frequency / density infection | `add_infection_*_flow` | `ForceOfInfection(kind=...)` or `EpiModel.add_infection_*_flow` |
| Mixing matrix | `Stratification.set_mixing_matrix` | `MixingMatrix` / `EpiModel.set_mixing_matrix` |
| Infectiousness by stratum | `add_infectiousness_adjustments` | `ForceOfInfection(infectiousness=...)` |
| Named parameter | `Parameter("beta")` | `Param("beta")` |
| Inspect $\lambda$ | derived output | `GroupedOutput("infection")` → `dims=("time", "age")` |

For a full age-stratified SEIRS with calibration, see the case study under
`docs/case-studies/age-stratified-seirs`.
